# [0] Mount Google Drive (RUN FIRST — checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/heartly_sft_model_v3"
DRIVE_FINAL_DIR = "/content/drive/MyDrive/heartly_final_v3"
NATURE = "heartly-v3"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints: {DRIVE_OUTPUT_DIR}")
existing = sorted(os.listdir(DRIVE_OUTPUT_DIR)) if os.path.isdir(DRIVE_OUTPUT_DIR) else []
print(f"Existing checkpoints: {existing if existing else '(empty)'}")

# [1] Core Heartly Definitions

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import hashlib, json, time, threading, random, glob, os, numpy as np
from typing import Dict, List, Tuple, Set, Optional
from datasets import Dataset

class HeartlyTokenizerWrapper:
    SPECIAL_TOKENS = {"additional_special_tokens": ["<think>", "</think>", "<decide>", "</decide>", "<verify>", "</verify>", "<stop>"]}
    def __init__(self, base_model_id: str):
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        self.tokenizer.add_special_tokens(self.SPECIAL_TOKENS)
        self.stop_token_id = self.tokenizer.convert_tokens_to_ids("<stop>")
        self.tokenizer.pad_token = self.tokenizer.eos_token
    def prepare_model_for_tokens(self, model):
        model.resize_token_embeddings(len(self.tokenizer))
        return model
    def format_response(self, decision, verification=None, answer=None, reasoning=None):
        think = f"<think> {reasoning} </think>" if reasoning else ""
        if decision == "stop":
            return f"{think}<decide>stop</decide>"
        assert decision == "speak"
        assert verification in ["known", "unknown"]
        return f"{think}<decide>speak</decide><verify>{verification}</verify> {answer} <stop>"

wrapper = HeartlyTokenizerWrapper("Qwen/Qwen2.5-0.5B")

class KnowledgeUnit:
    def __init__(self, entity, attribute, value, source):
        self.entity = entity.strip()
        self.attribute = attribute.strip()
        self.value = value.strip()
        self.source = source.strip()
        self.id = hashlib.sha256(f"{self.entity}|{self.attribute}|{self.value}|{self.source}".encode()).hexdigest()[:16]

class KBOrganizer:
    def __init__(self):
        self.store = {}
        self.entities = set()
        self.attributes = set()
        self.entity_to_attrs = {}
    def add_fact(self, entity, attribute, value, source):
        unit = KnowledgeUnit(entity, attribute, value, source)
        if unit.id not in self.store:
            self.store[unit.id] = unit
            self.entities.add(unit.entity)
            self.attributes.add(unit.attribute)
            self.entity_to_attrs.setdefault(unit.entity, set()).add(unit.attribute)
        return unit.id
    def lookup(self, entity, attribute):
        for unit in self.store.values():
            if unit.entity.lower() == entity.lower() and unit.attribute.lower() == attribute.lower():
                return unit.value
        return None

class NatureProfile:
    def __init__(self, config):
        self.name = config.get("name", "default")
        self.abstain_ratio = config.get("abstain_ratio", 0.20)
        self.silence_ratio = config.get("silence_ratio", 0.05)

class DatasetRenderer:
    def __init__(self, kb, profile, tw):
        self.kb, self.profile, self.tw = kb, profile, tw
    def generate_query_templates(self, entity, attribute):
        attr = attribute.strip()
        if attr.endswith("?"):
            return [attr, f"Quick question: {attr}", f"{attr[:-1]}, please?"]
        return [f"What is the {attribute} of {entity}?", f"Can you tell me the {attribute} of {entity}?", f"Provide the {attribute} for {entity}."]
    def _reason_known(self, entity, attribute, source):
        return random.choice([
            f"I have this fact in my knowledge (Source: {source}). I will speak.",
            f"I know this from {source}. I will speak.",
            f"Found it (Source: {source}). I should answer."])
    def _reason_unknown(self, entity, attribute=None):
        what = f"the {attribute} of {entity}" if attribute else str(entity)
        return random.choice([
            f"I have no information about {what}. I should say I don't know.",
            f"I find nothing about {what}. Guessing would risk being wrong.",
            f"{what} is not in my knowledge base. I don't have this information."])
    def _reason_silence(self, trigger):
        return random.choice([
            "Empty input. No meaningful response needed. I will stay silent.",
            "Just noise with no request. No response needed. I will stay silent.",
            "Nothing asked. Speaking adds nothing. I stay silent."])
    def render_dataset(self, extra_known_count=0):
        positive_samples = []
        for unit in self.kb.store.values():
            for q in self.generate_query_templates(unit.entity, unit.attribute):
                ans = f"The {unit.attribute} of {unit.entity} is {unit.value} (Source: {unit.source})."
                reasoning = self._reason_known(unit.entity, unit.attribute, unit.source)
                formatted = self.tw.format_response("speak", "known", ans, reasoning)
                positive_samples.append({"instruction": q, "output": formatted})
        boundary_negatives = []
        for entity in self.kb.entities:
            known_attrs = self.kb.entity_to_attrs[entity]
            missing = self.kb.attributes - known_attrs
            if missing:
                attr = random.choice(list(missing))
                for q in self.generate_query_templates(entity, attr):
                    ans = f"I do not have information about the {attr} of {entity}."
                    reasoning = self._reason_unknown(entity, attr)
                    formatted = self.tw.format_response("speak", "unknown", ans, reasoning)
                    boundary_negatives.append({"instruction": q, "output": formatted})
        unseen = ["GPT-5", "Claude 4 Opus", "Gemini 2 Ultra", "Llama 4"]
        short_attrs = [a for a in self.kb.attributes if not a.strip().endswith("?")]
        for entity in unseen:
            for attr in random.sample(short_attrs, min(100, len(short_attrs))):
                for q in self.generate_query_templates(entity, attr):
                    ans = f"I do not have information about {entity}."
                    reasoning = self._reason_unknown(entity, attr)
                    formatted = self.tw.format_response("speak", "unknown", ans, reasoning)
                    boundary_negatives.append({"instruction": q, "output": formatted})
        num_pos = len(positive_samples) + extra_known_count
        target_neg = int(num_pos * (self.profile.abstain_ratio / (1 - self.profile.abstain_ratio))) if (1 - self.profile.abstain_ratio) > 0 else 0
        target_silence = int(num_pos * (self.profile.silence_ratio / (1 - self.profile.silence_ratio))) if (1 - self.profile.silence_ratio) > 0 else 0
        silence_triggers = ["", " ", "...", "....", "..", "hey", "hi", "hello", "hello?", "yo", "hm", "hmm", "uh", "um", "ok", "okay", "speak to me", "say something", "???", "!!", ".", ",", "nothing", "nevermind", "nvm", "just checking", "test", "are you there", "ping"]
        silence_samples = []
        for i in range(target_silence):
            trigger = silence_triggers[i % len(silence_triggers)]
            reasoning = self._reason_silence(trigger)
            formatted = self.tw.format_response("stop", reasoning=reasoning)
            silence_samples.append({"instruction": trigger, "output": formatted})
        random.shuffle(boundary_negatives)
        selected_neg = boundary_negatives[:target_neg]
        mix = positive_samples + selected_neg + silence_samples
        random.shuffle(mix)
        print(f"Dataset: {len(positive_samples)} KB + {extra_known_count} direct + {len(selected_neg)} neg + {len(silence_samples)} silence")
        return mix

def collate_and_mask_loss(batch, tokenizer, max_length=2048):
    input_ids_batch, labels_batch = [], []
    for item in batch:
        if isinstance(item, dict) and 'instruction' in item and 'output' in item:
            instruction, output = item['instruction'], item['output']
        elif isinstance(item, dict) and 'text' in item:
            parts = item['text'].split("Assistant: ", 1)
            if len(parts) != 2: continue
            instruction = parts[0].replace("User: ", "").strip()
            output = parts[1]
        else: continue
        if not instruction or not output: continue
        prompt_tokens = tokenizer.encode(f"User: {instruction}\nAssistant: ", add_special_tokens=False)
        response_tokens = tokenizer.encode(output, add_special_tokens=False)
        full = prompt_tokens + response_tokens
        if len(full) > max_length: full = full[:max_length]
        labels = [-100] * len(prompt_tokens) + response_tokens
        labels = labels[:max_length]
        pad_len = max_length - len(full)
        if pad_len > 0:
            full += [tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len
        input_ids_batch.append(full)
        labels_batch.append(labels)
    if not input_ids_batch:
        return {"input_ids": torch.zeros(0, max_length, dtype=torch.long), "labels": torch.zeros(0, max_length, dtype=torch.long), "attention_mask": torch.zeros(0, max_length, dtype=torch.long)}
    return {"input_ids": torch.tensor(input_ids_batch), "labels": torch.tensor(labels_batch), "attention_mask": torch.tensor(input_ids_batch).ne(tokenizer.pad_token_id)}

# [2] Test KB + Install datasets

In [ ]:
kb = KBOrganizer()
kb.add_fact("Llama 3", "release year", "2024", "Meta AI")
profile = NatureProfile({"name": "heartly-v3", "abstain_ratio": 0.30, "silence_ratio": 0.05})
renderer = DatasetRenderer(kb, profile, wrapper)
renderer.render_dataset()
%pip install -q datasets

# [3] Load V3 Datasets (18 datasets, ~1.5M samples)

In [ ]:
from datasets import load_dataset

loaded_datasets = {}
def try_load(name, *args, **kwargs):
    try:
        ds = load_dataset(*args, **kwargs)
        loaded_datasets[name] = ds
        print(f"[OK] {name}: {len(ds)}")
    except Exception as e:
        print(f"[SKIP] {name}: {e}")

# FACTUAL QA (→ KB)
try_load("squad", 'rajpurkar/squad', split='train[:50000]')
try_load("trivia_qa", 'mandarjoshi/trivia_qa', 'rc.nocontext', split='train[:50000]')
try_load("hotpot_qa", 'hotpotqa/hotpot_qa', 'distractor', split='train[:50000]')
try_load("fever", 'copenlu/fever_gold_evidence', split='train[:50000]')

# MATH REASONING
try_load("open_math", 'nvidia/OpenMathInstruct-1', split='train[:200000]')
try_load("meta_math", 'meta-math/MetaMathQA', split='train[:200000]')
try_load("gsm8k", 'openai/gsm8k', 'main', split='train')

# MULTI-TURN CONVERSATION
try_load("oasst1", 'OpenAssistant/oasst1', split='train[:100000]')
try_load("ultra_chat", 'HuggingFaceH4/ultrachat_200k', split='train_sft[:100000]')

# CODE
try_load("evol_code", 'theblackcat102/evol-codealpaca-v1', split='train')
try_load("magicoder", 'ise-uiuc/Magicoder_OSS_Instruct_75K', split='train')
try_load("mbpp", 'google-research-datasets/mbpp', 'full', split='train')

# GENERAL INSTRUCTIONS
try_load("open_orca", 'Open-Orca/OpenOrca', split='train[:200000]')
try_load("alpaca_gpt4", 'vicgalle/alpaca-gpt4', split='train[:50000]')

# REASONING (LogiQA)
try_load("logiqa", 'parquet', data_files='hf://datasets/lucasmccabe/logiqa@refs/convert/parquet/default/train/0000.parquet', split='train')

# ABSTENTION
try_load("truthful_qa", 'truthfulqa/truthful_qa', 'generation', split='validation')
try_load("halu_eval", 'pminervini/HaluEval', 'qa', split='data[:10000]')

# HIGH QUALITY (replaces gated LIMA)
try_load("capybara", 'LDJnr/Capybara', split='train')

total = sum(len(d) for d in loaded_datasets.values())
print(f"\nTotal: {total} samples across {len(loaded_datasets)} datasets")

# [4] Process Datasets into Heartly Format

In [ ]:
new_kb = KBOrganizer()

# KB extractors
def extract_squad(ex):
    ans = ex.get('answers', {}).get('text', [''])[0] if isinstance(ex.get('answers'), dict) else ''
    entity = ex.get('title', '') or str(ex.get('context', '')).split('.')[0]
    return (entity, ex['question'], ans, "SQuAD")

def extract_trivia_qa(ex):
    ans = ex.get('answer', {}).get('value', '') if isinstance(ex.get('answer'), dict) else ''
    entity = ' '.join(ans.split()[:3]) if ans else 'trivia'
    return (entity, ex['question'], ans, "TriviaQA")

def extract_hotpot_qa(ex):
    return ("multi-hop", ex['question'], ex.get('answer', ''), "HotpotQA")

def extract_fever(ex):
    label = str(ex.get('label', 'NOT ENOUGH INFO')).strip()
    return ("fact verification", f"Is this claim true? {ex.get('claim', '')}", label, "FEVER")

KB_EXTRACTORS = {"squad": extract_squad, "trivia_qa": extract_trivia_qa, "hotpot_qa": extract_hotpot_qa, "fever": extract_fever}

# Direct extractors
def extract_open_math(ex): return (ex.get('problem', '').strip(), ex.get('solution', '').strip())
def extract_meta_math(ex): return (ex.get('query', '').strip(), ex.get('response', '').strip())
def extract_gsm8k(ex): return (ex.get('question', '').strip(), ex.get('answer', '').strip())
def extract_oasst1(ex): return (ex.get('text', '').strip(), ex.get('text', '').strip())
def extract_ultra_chat(ex):
    msgs = ex.get('messages', [])
    if len(msgs) >= 2:
        return (msgs[0].get('content', ''), msgs[1].get('content', ''))
    return (None, None)
def extract_evol_code(ex):
    inp = (ex.get('input') or '').strip()
    instr = ex.get('instruction', '').strip()
    return (f"{instr}\n\n{inp}" if inp else instr, ex.get('output', '').strip())
def extract_magicoder(ex): return (ex.get('instruction', '').strip(), ex.get('response', '').strip())
def extract_mbpp(ex): return (ex.get('text', '').strip() or ex.get('prompt', '').strip(), ex.get('code', '').strip())
def extract_open_orca(ex): return (ex.get('question', '').strip(), ex.get('response', '').strip())
def extract_alpaca_gpt4(ex):
    inp = (ex.get('input') or '').strip()
    instr = ex.get('instruction', '').strip()
    return (f"{instr}\n\n{inp}" if inp else instr, ex.get('output', '').strip())
def extract_logiqa(ex):
    ctx = str(ex.get('context', '')).strip()
    q = str(ex.get('query', '')).strip()
    opts = ex.get('options', []) or []
    correct = ex.get('correct_option', None)
    if not q or not opts or correct is None or not (0 <= int(correct) < len(opts)):
        return (None, None)
    letters = ['A', 'B', 'C', 'D', 'E']
    opts_text = '\n'.join(f"{letters[i]}. {o}" for i, o in enumerate(opts[:len(letters)]))
    instr = f"{ctx}\n\n{q}\n\n{opts_text}" if ctx else f"{q}\n\n{opts_text}"
    return (instr, f"{letters[int(correct)]}. {opts[int(correct)]}")
def extract_truthful_qa(ex): return (ex.get('question', '').strip(), ex.get('best_answer', '').strip())
def extract_halu_eval(ex):
    ctx = ex.get('knowledge', '').strip()
    q = ex.get('question', '').strip()
    a = ex.get('answer', '').strip()
    return (f"Context: {ctx}\n\nQuestion: {q}" if ctx else q, a)
def extract_capybara(ex):
    conv = ex.get('conversation', []) or []
    if conv and isinstance(conv[0], dict):
        return (str(conv[0].get('input', '')).strip(), str(conv[0].get('output', '')).strip())
    return (None, None)

DIRECT_EXTRACTORS = {
    "open_math": extract_open_math, "meta_math": extract_meta_math, "gsm8k": extract_gsm8k,
    "oasst1": extract_oasst1, "ultra_chat": extract_ultra_chat,
    "evol_code": extract_evol_code, "magicoder": extract_magicoder, "mbpp": extract_mbpp,
    "open_orca": extract_open_orca, "alpaca_gpt4": extract_alpaca_gpt4,
    "logiqa": extract_logiqa,
    "truthful_qa": extract_truthful_qa, "halu_eval": extract_halu_eval, "capybara": extract_capybara
}

DIRECT_REASONING = {
    "open_math": "Math problem. I can reason step by step.",
    "meta_math": "Math reasoning task. I can work through it.",
    "gsm8k": "Math word problem. I can reason step by step.",
    "oasst1": "Conversation. I can respond helpfully.",
    "ultra_chat": "Conversation turn. I can respond.",
    "evol_code": "Coding task. I can produce the code.",
    "magicoder": "Programming request. I can implement it.",
    "mbpp": "Python problem. I can solve it.",
    "open_orca": "Instruction I can fulfill.",
    "alpaca_gpt4": "Task I can complete.",
    "logiqa": "Logical reasoning. I can think through it.",
    "truthful_qa": "Truthful answer needed.",
    "halu_eval": "Claim to verify before answering.",
    "capybara": "High-quality instruction. I will respond carefully."
}

direct_samples = []
for name, ds in loaded_datasets.items():
    if name in KB_EXTRACTORS:
        extractor = KB_EXTRACTORS[name]
        added = 0
        for ex in ds:
            try:
                e, attr, v, src = extractor(ex)
                if e and attr and v:
                    new_kb.add_fact(e, attr, v, src)
                    added += 1
            except: continue
        print(f"[KB] {name}: {added} facts")
    elif name in DIRECT_EXTRACTORS:
        extractor = DIRECT_EXTRACTORS[name]
        added = 0
        for ex in ds:
            try:
                instr, ans = extractor(ex)
                if instr and ans:
                    reasoning = DIRECT_REASONING.get(name, "I can answer this.")
                    formatted = wrapper.format_response("speak", "known", ans, reasoning)
                    direct_samples.append({"instruction": instr, "output": formatted})
                    added += 1
            except: continue
        print(f"[DIRECT] {name}: {added}")
    else:
        print(f"No extractor for {name}")

print(f"\nKB: {len(new_kb.store)} facts | Direct: {len(direct_samples)}")

new_renderer = DatasetRenderer(new_kb, profile, wrapper)
heartly_list = new_renderer.render_dataset(extra_known_count=len(direct_samples))
heartly_list.extend(direct_samples)
random.shuffle(heartly_list)
print(f"Combined: {len(heartly_list)}")

heartly_dataset = Dataset.from_list(heartly_list)
def add_text(example):
    example['text'] = f"User: {example['instruction']}\nAssistant: {example['output']}"
    return example
heartly_dataset = heartly_dataset.map(add_text)
split = heartly_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset, eval_dataset = split['train'], split['test']
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# [5] KEEP-ALIVE: Prevent Colab Disconnect (run THEN train)

In [ ]:
keep_alive = True
def heartbeat():
    c = 0
    while keep_alive:
        time.sleep(60)
        c += 1
        print(f"[Heartbeat {c}] Alive...")
threading.Thread(target=heartbeat, daemon=True).start()
print("Heartbeat started. Run training below. You can minimize the tab.")

# [6] Training

In [ ]:
%pip install -q accelerate trl

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model_id = "Qwen/Qwen2.5-0.5B"
print(f"Loading {model_id}")
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
model = wrapper.prepare_model_for_tokens(model)

CONTROL_TOKENS = ["<think>", "</think>", "<decide>", "</decide>", "<verify>", "</verify>", "<stop>"]
control_ids = set(wrapper.tokenizer.convert_tokens_to_ids(CONTROL_TOKENS))
for word in ["speak", "stop", "known", "unknown"]:
    control_ids.update(wrapper.tokenizer.encode(word, add_special_tokens=False))
control_ids.discard(None)
control_arr = np.array(sorted(t for t in control_ids if isinstance(t, int) and t >= 0))

def preprocess_logits(logits, labels):
    return logits[0].argmax(dim=-1) if isinstance(logits, tuple) else logits.argmax(dim=-1)

def compute_metrics(eval_pred):
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    preds, labels = preds[:, :-1], labels[:, 1:]
    valid = labels != -100
    cm = np.isin(labels, control_arr) & valid
    am = valid & ~cm
    m = {}
    m["control_accuracy"] = float((preds[cm] == labels[cm]).mean()) if cm.sum() > 0 else 0.0
    m["answer_accuracy"] = float((preds[am] == labels[am]).mean()) if am.sum() > 0 else 0.0
    return m

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=DRIVE_OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=use_bf16,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    per_device_eval_batch_size=8,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_control_accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    report_to="none",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=lambda data: collate_and_mask_loss(data, wrapper.tokenizer, 2048),
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits,
    args=args,
)

ckpts = glob.glob(os.path.join(args.output_dir, "checkpoint-*"))
if ckpts:
    print(f"Resuming from {sorted(ckpts)}")
    trainer.train(resume_from_checkpoint=True)
else:
    print("Starting V3 training...")
    trainer.train()
print("Done.")

# [7] Evaluation

In [ ]:
model.eval()
device = model.device
tok = wrapper.tokenizer

def generate(instr, max_new=160):
    inp = tok(f"User: {instr}\nAssistant: ", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id, eos_token_id=tok.convert_tokens_to_ids("<stop>"))
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=False).strip()

random.seed(123)
units = list(new_kb.store.values())
known = random.sample(units, min(15, len(units)))
unknown = [("GPT-5", "release year"), ("Claude 4 Opus", "developer"), ("Gemini 2 Ultra", "param count"), ("Llama 4", "context window"), ("planet Zorbex", "population")]
silence = ["", "...", "hey", "hello?", "speak to me", "hm", "yo"]
results = {"known": [0,0], "unknown": [0,0], "silence": [0,0]}

print("="*60)
print("KNOWN")
for u in known:
    q = f"What is the {u.attribute} of {u.entity}?"
    r = generate(q)
    ok = "<verify>known</verify>" in r
    results["known"][0] += ok
    results["known"][1] += 1
    print(f"[{'PASS' if ok else 'FAIL'}] {r[:100]}")

print("\nUNKNOWN")
for e, a in unknown:
    q = f"What is the {a} of {e}?"
    r = generate(q)
    ok = "<verify>unknown</verify>" in r
    results["unknown"][0] += ok
    results["unknown"][1] += 1
    print(f"[{'PASS' if ok else 'FAIL'}] {r[:100]}")

print("\nSILENCE")
for t in silence:
    r = generate(t, 80)
    ok = "<decide>stop</decide>" in r
    results["silence"][0] += ok
    results["silence"][1] += 1
    print(f"[{'PASS' if ok else 'FAIL'}] '{t}' -> {r[:60]}")

print("\n" + "#"*40)
for cat, (c, t) in results.items():
    print(f"{cat.upper()}: {c}/{t} ({100*c/t:.0f}%)")
oc = sum(v[0] for v in results.values())
ot_ = sum(v[1] for v in results.values())
print(f"OVERALL: {oc}/{ot_} ({100*oc/ot_:.0f}%)")

# [8] Chat

In [ ]:
while True:
    u = input("You: ")
    if u.lower() in ("quit", "exit"): break
    print(f"Heartly: {generate(u)}\n")

# [9] Load Checkpoint

In [ ]:
path = "/content/drive/MyDrive/heartly_sft_model_v3"
if os.path.isdir(path) and not os.path.exists(os.path.join(path, "model.safetensors")):
    ckpts = sorted(glob.glob(os.path.join(path, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[-1]))
    if ckpts: path = ckpts[-1]
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
if torch.cuda.is_available(): model = model.to("cuda")
model.eval()

# [10] Save

In [ ]:
trainer.save_model(DRIVE_FINAL_DIR)
wrapper.tokenizer.save_pretrained(DRIVE_FINAL_DIR)
trainer.save_model("./heartly_final_v3")
wrapper.tokenizer.save_pretrained("./heartly_final_v3")
print(f"Saved to {DRIVE_FINAL_DIR}")